# Notebook 14 — Research Summary & Production Model Packaging

This notebook is the final research handoff stage.

It collects the frozen configuration and final untouched-test results from Notebooks 12–13 and packages the selected model for downstream application/API work.

### This notebook does not retune the model.

The research configuration is already frozen.

### Outputs

- Final research summary
- Model-performance summary
- Strategy-performance summary
- Research limitations
- Frozen model artifact
- Feature metadata
- Frozen configuration
- Production prediction schema
- Model manifest
- Production-ready artifact directory
- Final research package report


In [ ]:
from pathlib import Path
import json
import shutil
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGBClassifier = None
    XGB_AVAILABLE = False

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if not (ROOT / "data").exists():
    for c in [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/quant-trading-research"),
    ]:
        if (c / "data").exists() and (c / "notebooks").exists():
            ROOT = c
            break

MASTER_PATH = ROOT / "data" / "raw" / "sp500_1950_present.csv"
CONFIG_PATH = ROOT / "models" / "frozen" / "sp500_frozen_research_config.json"
METADATA_PATH = ROOT / "models" / "frozen" / "sp500_frozen_model_metadata.json"
FINAL_REPORT_PATH = ROOT / "reports" / "generated" / "sp500_final_untouched_test_report.json"

PRODUCTION_DIR = ROOT / "models" / "production"
REPORT_DIR = ROOT / "reports" / "generated"
TABLE_DIR = ROOT / "reports" / "tables"

for path in [
    PRODUCTION_DIR,
    REPORT_DIR,
    TABLE_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Production artifact directory:", PRODUCTION_DIR)


In [ ]:
required_paths = [
    CONFIG_PATH,
    METADATA_PATH,
    FINAL_REPORT_PATH,
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Required Notebook 12/13 artifacts are missing:\n" +
        "\n".join(missing)
    )

frozen_config = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

frozen_metadata = json.loads(
    METADATA_PATH.read_text(encoding="utf-8")
)

final_report = json.loads(
    FINAL_REPORT_PATH.read_text(encoding="utf-8")
)

print("Frozen configuration:")
print(json.dumps(frozen_config, indent=2))

print("\nFinal research status:")
print(final_report.get("status"))
print("Research verdict:", final_report.get("research_verdict"))


In [ ]:
raw = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

raw["Date"] = pd.to_datetime(
    raw["Date"],
    errors="coerce"
)

for column in [
    "Open",
    "High",
    "Low",
    "Close",
    "Adj.Close",
    "Volume",
]:
    raw[column] = pd.to_numeric(
        raw[column],
        errors="coerce"
    )

raw = (
    raw
    .sort_values("Date")
    .reset_index(drop=True)
)

assert raw["Date"].notna().all()
assert raw["Date"].is_unique
assert raw["Date"].is_monotonic_increasing

print("Raw rows:", len(raw))
print(
    "Date range:",
    raw["Date"].min().date(),
    "→",
    raw["Date"].max().date()
)


In [ ]:
def build_features(df):
    data = df.copy()

    data["return_1d"] = data["Close"].pct_change()
    data["return_5d"] = data["Close"].pct_change(5)
    data["return_21d"] = data["Close"].pct_change(21)
    data["return_63d"] = data["Close"].pct_change(63)
    data["return_126d"] = data["Close"].pct_change(126)
    data["return_252d"] = data["Close"].pct_change(252)

    data["volatility_5d"] = (
        data["return_1d"].rolling(5).std()
        * np.sqrt(252)
    )

    data["volatility_21d"] = (
        data["return_1d"].rolling(21).std()
        * np.sqrt(252)
    )

    data["volatility_63d"] = (
        data["return_1d"].rolling(63).std()
        * np.sqrt(252)
    )

    data["sma_20"] = data["Close"].rolling(20).mean()
    data["sma_50"] = data["Close"].rolling(50).mean()
    data["sma_200"] = data["Close"].rolling(200).mean()

    data["price_to_sma_20"] = (
        data["Close"] / data["sma_20"] - 1
    )

    data["price_to_sma_50"] = (
        data["Close"] / data["sma_50"] - 1
    )

    data["price_to_sma_200"] = (
        data["Close"] / data["sma_200"] - 1
    )

    data["sma_50_vs_sma_200"] = (
        data["sma_50"] / data["sma_200"] - 1
    )

    data["range_pct"] = (
        (data["High"] - data["Low"]) /
        data["Close"]
    )

    data["intraday_return"] = (
        data["Close"] / data["Open"] - 1
    )

    data["overnight_return"] = (
        data["Open"] /
        data["Close"].shift(1) - 1
    )

    data["volume_ratio_20"] = (
        data["Volume"] /
        data["Volume"].rolling(20).mean()
    )

    data["volume_ratio_63"] = (
        data["Volume"] /
        data["Volume"].rolling(63).mean()
    )

    data["return_1d_lag1"] = (
        data["return_1d"].shift(1)
    )
    data["return_1d_lag2"] = (
        data["return_1d"].shift(2)
    )
    data["return_1d_lag3"] = (
        data["return_1d"].shift(3)
    )
    data["return_1d_lag5"] = (
        data["return_1d"].shift(5)
    )
    data["return_1d_lag10"] = (
        data["return_1d"].shift(10)
    )

    data["volatility_21d_lag1"] = (
        data["volatility_21d"].shift(1)
    )

    data["range_pct_lag1"] = (
        data["range_pct"].shift(1)
    )

    data["volume_ratio_20_lag1"] = (
        data["volume_ratio_20"].shift(1)
    )

    data["next_day_return"] = (
        data["Close"].shift(-1) /
        data["Close"] - 1
    )

    data["target"] = (
        data["next_day_return"] > 0
    ).astype(int)

    feature_columns = [
        "return_1d",
        "return_5d",
        "return_21d",
        "return_63d",
        "return_126d",
        "return_252d",
        "volatility_5d",
        "volatility_21d",
        "volatility_63d",
        "price_to_sma_20",
        "price_to_sma_50",
        "price_to_sma_200",
        "sma_50_vs_sma_200",
        "range_pct",
        "intraday_return",
        "overnight_return",
        "volume_ratio_20",
        "volume_ratio_63",
        "return_1d_lag1",
        "return_1d_lag2",
        "return_1d_lag3",
        "return_1d_lag5",
        "return_1d_lag10",
        "volatility_21d_lag1",
        "range_pct_lag1",
        "volume_ratio_20_lag1",
    ]

    return data, feature_columns


featured, feature_columns = build_features(raw)

print("Feature count:", len(feature_columns))
print("Features:")
for feature in feature_columns:
    print(" -", feature)


In [ ]:
candidate_model = frozen_config["candidate_model"]

def create_model(name):
    if name == "Logistic Regression":
        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    random_state=42
                )
            ),
        ])

    if name == "Random Forest":
        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=250,
                    max_depth=8,
                    min_samples_leaf=10,
                    max_features="sqrt",
                    random_state=42,
                    n_jobs=-1
                )
            ),
        ])

    if name == "HistGradientBoosting":
        return Pipeline([
            (
                "model",
                HistGradientBoostingClassifier(
                    max_iter=250,
                    learning_rate=0.05,
                    max_leaf_nodes=15,
                    l2_regularization=1.0,
                    random_state=42
                )
            ),
        ])

    if name == "XGBoost":
        if not XGB_AVAILABLE:
            raise ImportError(
                "XGBoost is required to package the frozen candidate model."
            )

        return XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.80,
            colsample_bytree=0.80,
            min_child_weight=5,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

    raise ValueError(
        f"Unsupported candidate model: {name}"
    )


production_model = create_model(candidate_model)

training_data = featured.dropna(
    subset=feature_columns + [
        "next_day_return",
        "target"
    ]
).copy()

production_model.fit(
    training_data[feature_columns],
    training_data["target"]
)

print(
    "Production packaging model fitted on all available "
    "historical labeled observations."
)

print("Training rows:", len(training_data))
print(
    "Training range:",
    training_data["Date"].min().date(),
    "→",
    training_data["Date"].max().date()
)


In [ ]:
import joblib

MODEL_PATH = (
    PRODUCTION_DIR /
    "sp500_direction_model.joblib"
)

joblib.dump(
    production_model,
    MODEL_PATH
)

print("Saved model:", MODEL_PATH)
print("Size:", MODEL_PATH.stat().st_size, "bytes")


In [ ]:
production_metadata = {
    "artifact_name": "sp500_direction_model",
    "model_type": candidate_model,
    "task": "next_day_direction_classification",
    "target_definition": "1 if next_day_return > 0 else 0",
    "feature_columns": feature_columns,
    "feature_count": len(feature_columns),
    "probability_threshold": frozen_config[
        "probability_threshold"
    ],
    "signal_type": frozen_config[
        "signal_type"
    ],
    "transaction_cost_bps": frozen_config[
        "transaction_cost_bps"
    ],
    "slippage_bps": frozen_config[
        "slippage_bps"
    ],
    "training_rows": len(training_data),
    "training_start": training_data[
        "Date"
    ].min().strftime("%Y-%m-%d"),
    "training_end": training_data[
        "Date"
    ].max().strftime("%Y-%m-%d"),
    "research_verdict": final_report.get(
        "research_verdict"
    ),
    "final_test_metrics": final_report.get(
        "classification_metrics"
    ),
    "final_test_strategy_metrics": final_report.get(
        "strategy_metrics"
    ),
    "warning": (
        "This artifact is packaged from a research configuration. "
        "Historical performance does not guarantee future performance."
    )
}

METADATA_OUT = (
    PRODUCTION_DIR /
    "sp500_direction_model_metadata.json"
)

METADATA_OUT.write_text(
    json.dumps(
        production_metadata,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        production_metadata,
        indent=2,
        default=str
    )
)


In [ ]:
# Copy the frozen research configuration into the production artifact directory.

CONFIG_OUT = (
    PRODUCTION_DIR /
    "sp500_frozen_research_config.json"
)

CONFIG_OUT.write_text(
    json.dumps(
        frozen_config,
        indent=2
    ),
    encoding="utf-8"
)

print("Saved:", CONFIG_OUT)


In [ ]:
# Production input schema.

input_schema = {
    "required_columns": [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Adj.Close",
        "Volume"
    ],
    "description": (
        "The prediction service should receive chronological S&P 500 "
        "OHLCV history and generate the latest feature row using the "
        "same feature-generation logic used during research."
    ),
    "minimum_history_required": 253,
    "output_columns": [
        "Date",
        "probability_up",
        "prediction",
        "signal"
    ],
    "signal_definition": {
        "prediction": "1 when probability_up >= threshold",
        "signal": "LONG when prediction=1, otherwise CASH"
    }
}

SCHEMA_OUT = (
    PRODUCTION_DIR /
    "prediction_input_output_schema.json"
)

SCHEMA_OUT.write_text(
    json.dumps(
        input_schema,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        input_schema,
        indent=2
    )
)


In [ ]:
# Smoke-test the packaged artifact by reloading it.

loaded_model = joblib.load(
    MODEL_PATH
)

latest_valid = (
    featured[
        feature_columns
    ]
    .dropna()
    .tail(1)
)

assert len(latest_valid) == 1

latest_probability = float(
    loaded_model.predict_proba(
        latest_valid
    )[:, 1][0]
)

latest_prediction = int(
    latest_probability >=
    frozen_config["probability_threshold"]
)

latest_signal = (
    "LONG"
    if latest_prediction == 1
    else "CASH"
)

smoke_test = pd.DataFrame([{
    "probability_up": latest_probability,
    "prediction": latest_prediction,
    "signal": latest_signal
}])

display(smoke_test)

assert 0 <= latest_probability <= 1

print("Production artifact smoke test: PASS")


In [ ]:
# Create a complete artifact manifest.

artifact_files = [
    MODEL_PATH,
    METADATA_OUT,
    CONFIG_OUT,
    SCHEMA_OUT,
]

manifest = {
    "package_name": "S&P 500 Direction Research Model",
    "candidate_model": candidate_model,
    "files": [
        {
            "name": path.name,
            "relative_path": str(
                path.relative_to(ROOT)
            ),
            "bytes": path.stat().st_size
        }
        for path in artifact_files
    ],
    "frozen_configuration": frozen_config,
    "smoke_test": {
        "status": "PASS",
        "latest_probability_up": latest_probability,
        "latest_prediction": latest_prediction,
        "latest_signal": latest_signal
    },
    "research_report": {
        "verdict": final_report.get(
            "research_verdict"
        ),
        "final_test_start": final_report.get(
            "test_start"
        ),
        "final_test_end": final_report.get(
            "test_end"
        )
    }
}

MANIFEST_PATH = (
    PRODUCTION_DIR /
    "model_manifest.json"
)

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        manifest,
        indent=2,
        default=str
    )
)


In [ ]:
# Research summary table.

classification = final_report.get(
    "classification_metrics",
    {}
)

strategy = final_report.get(
    "strategy_metrics",
    {}
)

benchmark = final_report.get(
    "benchmark_metrics",
    {}
)

summary_table = pd.DataFrame([{
    "candidate_model": candidate_model,
    "final_test_ROC_AUC": classification.get("roc_auc"),
    "final_test_F1": classification.get("f1"),
    "final_test_accuracy": classification.get("accuracy"),
    "strategy_CAGR": strategy.get("CAGR"),
    "strategy_Sharpe": strategy.get("Sharpe"),
    "strategy_Sortino": strategy.get("Sortino"),
    "strategy_max_drawdown": strategy.get("max_drawdown"),
    "benchmark_CAGR": benchmark.get("CAGR"),
    "benchmark_Sharpe": benchmark.get("Sharpe"),
    "research_verdict": final_report.get(
        "research_verdict"
    )
}])

display(summary_table)

summary_table.to_csv(
    TABLE_DIR /
    "sp500_final_research_summary.csv",
    index=False
)


In [ ]:
# Create a human-readable research handoff.

handoff = f'''
# S&P 500 Quant Research — Final Handoff

## Candidate Model

{candidate_model}

## Frozen Trading Configuration

- Signal type: {frozen_config["signal_type"]}
- Probability threshold: {frozen_config["probability_threshold"]}
- Transaction cost: {frozen_config["transaction_cost_bps"]} bps
- Slippage: {frozen_config["slippage_bps"]} bps

## Final Untouched Test

- Start: {final_report.get("test_start")}
- End: {final_report.get("test_end")}
- ROC-AUC: {classification.get("roc_auc")}
- F1: {classification.get("f1")}
- Accuracy: {classification.get("accuracy")}

## Final Strategy

- CAGR: {strategy.get("CAGR")}
- Sharpe: {strategy.get("Sharpe")}
- Sortino: {strategy.get("Sortino")}
- Maximum drawdown: {strategy.get("max_drawdown")}

## Buy & Hold Benchmark

- CAGR: {benchmark.get("CAGR")}
- Sharpe: {benchmark.get("Sharpe")}
- Maximum drawdown: {benchmark.get("max_drawdown")}

## Research Verdict

{final_report.get("research_verdict")}

## Important Limitation

Historical backtests and statistical tests do not guarantee future profitability.
The packaged artifact should be monitored after deployment and periodically
revalidated using a separately controlled research process.
'''

HANDOFF_PATH = (
    REPORT_DIR /
    "sp500_final_research_handoff.md"
)

HANDOFF_PATH.write_text(
    handoff.strip() + "\n",
    encoding="utf-8"
)

print(handoff)
print("Saved:", HANDOFF_PATH)


In [ ]:
# Final production package validation.

for path in artifact_files + [
    MANIFEST_PATH,
]:
    assert path.exists()
    assert path.stat().st_size > 0

assert production_metadata["feature_count"] == len(feature_columns)
assert production_metadata["model_type"] == candidate_model
assert frozen_config["probability_threshold"] == (
    production_metadata["probability_threshold"]
)

print("Production package validation: PASS")
print("Artifact directory:", PRODUCTION_DIR)
print("Files:")
for path in sorted(PRODUCTION_DIR.iterdir()):
    print(" -", path.name)


In [ ]:
# Final raw master-data integrity check.

master_check = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

expected_columns = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Adj.Close",
    "Volume"
]

assert list(master_check.columns) == expected_columns

master_dates = pd.to_datetime(
    master_check["Date"],
    errors="coerce"
)

assert master_dates.notna().all()
assert master_dates.is_unique
assert master_dates.is_monotonic_increasing

print("Raw master dataset integrity: PASS")
print("Master rows:", len(master_check))


# Notebook 14 Complete

The research model is now packaged without changing the frozen configuration.

### Final package

- `sp500_direction_model.joblib`
- `sp500_direction_model_metadata.json`
- `sp500_frozen_research_config.json`
- `prediction_input_output_schema.json`
- `model_manifest.json`

### Research outputs

- Final research summary CSV
- Final research handoff Markdown
- Frozen configuration
- Final untouched-test report
- Production artifact smoke test

### Important boundary

The model artifact is packaged for downstream production engineering. It is **not** a claim that the strategy is guaranteed profitable.

The next stage can move from research notebooks into the application layer:
- prediction service
- FastAPI
- Streamlit dashboard
- model loading
- signal generation
- monitoring
- Docker
- deployment

Run Notebook 14 completely and verify the production artifacts before beginning the application/deployment phase.
